<a href="https://colab.research.google.com/github/arwindersingh1508-netizen/Delhivery-Last-Mile-Delay-Intelligence-Pipeline/blob/main/Delhivery_Last_Mile_Delay_Intelligence_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""Synthetic shipment data for the Delay Intelligence capstone.
Run once:  python generate_data.py   (writes data/hubs.csv, data/raw_shipments.csv, data/new_shipments.csv)
Do not edit any constant: every number in the brief was measured on this exact output."""
import os
import numpy as np
import pandas as pd

HUBS = [
    ("HUB01", "Mumbai", "West", 1, 1500), ("HUB02", "Delhi", "North", 1, 1600),
    ("HUB03", "Bengaluru", "South", 1, 1450), ("HUB04", "Hyderabad", "South", 1, 1350),
    ("HUB05", "Chennai", "South", 1, 1300), ("HUB06", "Pune", "West", 1, 1250),
    ("HUB07", "Kolkata", "East", 1, 1200), ("HUB08", "Ahmedabad", "West", 1, 1150),
    ("HUB09", "Jaipur", "North", 2, 950), ("HUB10", "Lucknow", "North", 2, 900),
    ("HUB11", "Chandigarh", "North", 2, 850), ("HUB12", "Indore", "West", 2, 820),
    ("HUB13", "Nagpur", "West", 2, 780), ("HUB14", "Coimbatore", "South", 2, 740),
    ("HUB15", "Kochi", "South", 2, 700), ("HUB16", "Surat", "West", 2, 650),
    ("HUB17", "Patna", "East", 3, 520), ("HUB18", "Ranchi", "East", 3, 480),
    ("HUB19", "Guwahati", "East", 3, 450), ("HUB20", "Raipur", "East", 3, 420),
    ("HUB21", "Bhubaneswar", "East", 3, 390), ("HUB22", "Dehradun", "North", 3, 350),
    ("HUB23", "Siliguri", "East", 3, 310), ("HUB24", "Agartala", "East", 3, 270),
]
DIST_MU = [180, 320, 480]      # mean lane distance (km) for hub tier 1 / 2 / 3
DIST_SD = [50, 90, 150]
STOPS_MU = [55, 70, 85]        # mean stops on the rider route for hub tier 1 / 2 / 3
HUB_EFFECT = [0, 2, 4]         # extra hours for hub tier 1 / 2 / 3
# How a hub really operates (1 = metro-like, 2 = regional, 3 = stretched). Four hubs operate unlike their tier label.
OPERATING_TIER = {"HUB07": 2, "HUB13": 3, "HUB15": 3, "HUB22": 2}
BASE_HOURS = {"Express": 10, "Standard": 28, "Economy": 58}


def simulate(n_per_hub, seed):
    """Clean shipments, n_per_hub rows for each of the 24 hubs, in hub order."""
    np.random.seed(seed)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    n = n_per_hub * len(hubs)
    op_tier = [OPERATING_TIER.get(h, t) for h, t in zip(hubs["hub_id"], hubs["hub_tier"])]
    tier = np.repeat(op_tier, n_per_hub) - 1
    hub_id = np.repeat(hubs["hub_id"].values, n_per_hub)
    distance = np.clip(np.random.normal(np.array(DIST_MU)[tier], np.array(DIST_SD)[tier]), 15, 1500).round(1)
    stops = np.clip(np.random.normal(np.array(STOPS_MU)[tier], 10), 20, 140).round().astype(int)
    weight = np.clip(np.random.lognormal(0.6, 0.7, n), 0.1, 40).round(2)
    value = np.random.lognormal(7.0, 0.8, n).round().astype(int)
    service = np.random.choice(["Express", "Standard", "Economy"], n, p=[0.2, 0.55, 0.25])
    pay = np.random.choice(["Prepaid", "COD"], n, p=[0.6, 0.4])
    rain = (np.random.rand(n) < 0.18).astype(int)
    day = np.random.randint(0, 90, n)
    hour = np.random.randint(6, 22, n)
    minute = np.random.randint(0, 60, n)
    ts = pd.Timestamp("2026-07-01") + pd.to_timedelta(day, unit="D") + pd.to_timedelta(hour * 60 + minute, unit="m")
    sale = ((ts >= pd.Timestamp("2026-09-15")) & (ts < pd.Timestamp("2026-09-29"))).astype(int)
    weekend = (ts.dayofweek >= 5).astype(int)
    hours = (np.array([BASE_HOURS[s] for s in service]) + 0.03 * distance + 0.10 * stops
             + np.array(HUB_EFFECT)[tier] + 0.12 * weight + 4 * rain + 7 * sale
             + 3 * (pay == "COD") + 2 * weekend + 3 * (hour >= 18)
             + 12 * sale * (service == "Express") + 8 * (distance > 400) * rain
             + np.random.normal(0, 4, n)).round(2)
    return pd.DataFrame({
        "hub_id": hub_id, "service_type": service, "payment_mode": pay,
        "parcel_weight_kg": weight, "distance_km": distance, "declared_value": value,
        "n_stops": stops, "rain_flag": rain, "sale_event_flag": sale, "dispatch_time": ts,
        "label_print_batch": np.random.randint(1000, 9999, n),
        "scanner_build_no": np.random.randint(1, 40, n),
        "tape_roll_no": np.random.randint(1, 500, n),
        "sticker_serial_mod": np.random.uniform(0, 1, n).round(3),
        "shelf_slot_no": np.random.randint(1, 300, n),
        "delivery_hours": hours,
    })


def make_messy(df, seed, n_missing, n_case, n_sentinel, n_dupes, id_start):
    """Turn a clean table into a raw export: text columns, gaps, duplicates, a sentinel."""
    np.random.seed(seed)
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n = len(df)
    df.insert(0, "shipment_id", ["SHP" + str(id_start + i) for i in range(n)])
    df["dispatch_time"] = df["dispatch_time"].dt.strftime("%Y-%m-%d %H:%M")
    df["declared_value"] = df["declared_value"].map(lambda v: "\u20b9" + format(v, ","))
    for col, k in [("distance_km", n_missing[0]), ("parcel_weight_kg", n_missing[1]),
                   ("declared_value", n_missing[2]), ("payment_mode", n_missing[3])]:
        df.loc[np.random.choice(n, k, replace=False), col] = np.nan
    pick = np.random.choice(n, n_case, replace=False)
    shapes = np.random.randint(0, 3, n_case)
    for i, s in zip(pick, shapes):
        v = df.at[i, "service_type"]
        df.at[i, "service_type"] = [v.upper(), v.lower(), "  " + v + " "][s]
    if n_sentinel:
        df.loc[np.random.choice(n, n_sentinel, replace=False), "parcel_weight_kg"] = 999.0
    if n_dupes:
        df = pd.concat([df, df.iloc[np.random.choice(n, n_dupes, replace=False)]], ignore_index=True)
    return df


if __name__ == "__main__":
    os.makedirs("data", exist_ok=True)
    hubs = pd.DataFrame(HUBS, columns=["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"])
    hubs.to_csv("data/hubs.csv", index=False)
    raw = make_messy(simulate(50, 2604), seed=7, n_missing=(36, 24, 12, 18),
                     n_case=120, n_sentinel=5, n_dupes=24, id_start=10001)
    raw.to_csv("data/raw_shipments.csv", index=False)
    new = make_messy(simulate(2, 99), seed=8, n_missing=(3, 2, 1, 2),
                     n_case=6, n_sentinel=0, n_dupes=0, id_start=90001)
    new = new.drop(columns=["delivery_hours"])
    new.to_csv("data/new_shipments.csv", index=False)
    print(len(hubs), len(raw), len(new))



24 1224 48


In [3]:
PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}
FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value", "n_stops",
    "rain_flag", "sale_event_flag", "is_cod", "is_weekend", "dispatch_hour",
    "hub_tier", "hub_daily_capacity",
    "label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no",
]
NOISE_COLUMNS = ["label_print_batch", "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no"]


In [1]:

!pip install pandas==2.2.3 numpy==2.1.3 scikit-learn==1.6.1 joblib==1.6.0


In [2]:
requirements = """
pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
joblib==1.6.0
"""

with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

print("done")

done


In [3]:
with open("requirements.txt", "r") as file:
    print(file.read())

pandas==2.2.3
numpy==2.1.3
scikit-learn==1.6.1
joblib==1.6.0


In [4]:
import os
import pandas as pd
import numpy as np

# 1. Create all required directories
os.makedirs("src", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

# 2. Create requirements.txt
with open("requirements.txt", "w") as f:
    f.write("pandas==2.2.3\nnumpy==1.26.4\nscikit-learn==1.5.2\njoblib==1.4.2\n")

# 3. Create src/__init__.py
with open("src/__init__.py", "w") as f:
    pass

print("Directories and base files created successfully!")

Directories and base files created successfully!


In [10]:
%%writefile src/shipment_prep.py
import pandas as pd
import numpy as np

PROMISED_HOURS = {"Express": 36, "Standard": 60, "Economy": 96}

FEATURES = [
    "promised_hours", "distance_km", "parcel_weight_kg", "log_declared_value",
    "n_stops", "rain_flag", "sale_event_flag", "is_cod", "is_weekend",
    "dispatch_hour", "hub_tier", "hub_daily_capacity", "label_print_batch",
    "scanner_build_no", "tape_roll_no", "sticker_serial_mod", "shelf_slot_no"
]

NOISE_COLUMNS = [
    "label_print_batch", "scanner_build_no", "tape_roll_no",
    "sticker_serial_mod", "shelf_slot_no"
]

def clean_shipments(raw, medians=None):
    """Takes a raw shipments DataFrame and returns a tuple (clean_df, medians)."""
    df = raw.copy()

    # 1. Drop exact duplicate rows
    df = df.drop_duplicates()

    # 2. service_type: strip leading/trailing spaces and convert to title case
    df["service_type"] = df["service_type"].str.strip().str.title()

    # 3. payment_mode: replace missing value with "Unknown"
    df["payment_mode"] = df["payment_mode"].fillna("Unknown")

    # 4. declared_value: remove rupee symbol and commas, convert to numeric
    if "declared_value" in df.columns:
        df["declared_value_inr"] = (
            df["declared_value"]
            .astype(str)
            .str.replace("\u20b9", "", regex=False)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df["declared_value_inr"] = pd.to_numeric(df["declared_value_inr"], errors="coerce")
        df = df.drop(columns=["declared_value"])

    # 5. dispatch_time: convert to datetime
    df["dispatch_time"] = pd.to_datetime(df["dispatch_time"], errors="coerce")

    # 6. parcel_weight_kg: values above 100 are scanner faults -> treat as missing
    if "parcel_weight_kg" in df.columns:
        df.loc[df["parcel_weight_kg"] > 100, "parcel_weight_kg"] = np.nan

    # 7. Fill missing values with medians
    cols_to_fill = ["distance_km", "parcel_weight_kg", "declared_value_inr"]
    computed_medians = {}

    if medians is None:
        for col in cols_to_fill:
            if col in df.columns:
                computed_medians[col] = float(df[col].median())
            else:
                computed_medians[col] = 0.0
    else:
        computed_medians = medians

    for col in cols_to_fill:
        if col in df.columns:
            df[col] = df[col].fillna(computed_medians[col])

    return df, computed_medians

def build_features(clean, hubs):
    """Returns a DataFrame with hub columns attached and new features engineered."""
    df = clean.copy()

    # Merge with hubs
    df = df.merge(
        hubs[["hub_id", "hub_city", "region", "hub_tier", "hub_daily_capacity"]],
        on="hub_id",
        how="left"
    )

    # Promised hours based on service type
    df["promised_hours"] = df["service_type"].map(PROMISED_HOURS)

    # is_cod
    df["is_cod"] = (df["payment_mode"] == "COD").astype(int)

    # is_weekend (1 when dispatch day is Saturday or Sunday)
    df["is_weekend"] = df["dispatch_time"].dt.dayofweek.isin([5, 6]).astype(int)

    # dispatch_hour
    df["dispatch_hour"] = df["dispatch_time"].dt.hour

    # log_declared_value
    df["log_declared_value"] = np.log(df["declared_value_inr"])

    return df


Writing src/shipment_prep.py


In [20]:
%%writefile part_a_clean_eda.py
import os
import sys
import numpy as np
import pandas as pd

from src.shipment_prep import clean_shipments, build_features, PROMISED_HOURS, FEATURES

# 1. Read data
raw = pd.read_csv("data/raw_shipments.csv")
hubs = pd.read_csv("data/hubs.csv")

# 2. Clean shipments
clean_df, computed_medians = clean_shipments(raw)

# Save impute values
impute_df = pd.DataFrame(list(computed_medians.items()), columns=["metric", "value"])
impute_df.to_csv("models/impute_values.csv", index=False)

# 3. Build features
model_df = build_features(clean_df, hubs)

# 4. Target: is_delayed (1 when delivery_hours > promised_hours, else 0)
model_df["is_delayed"] = (model_df["delivery_hours"] > model_df["promised_hours"]).astype(int)

# Save modelling table
model_df.to_csv("data/modelling_table.csv", index=False)

# 5. Checks file
checks_data = [
    {"metric": "total_shipments_modeled", "value": len(model_df)}
]
checks_df = pd.DataFrame(checks_data, columns=["metric", "value"])
checks_df.to_csv("outputs/part_a_checks.csv", index=False)

# 6. Delay evidence table
bins = [0, 200, 400, 600, float("inf")]
labels = ["0-200", "200-400", "400-600", "600+"]
model_df["distance_band"] = pd.cut(model_df["distance_km"], bins=bins, labels=labels, right=False)

delay_rows = []
for dim in ["service_type", "hub_tier", "distance_band"]:
    grouped = model_df.groupby(dim, observed=False).agg(
        shipments=("is_delayed", "count"),
        delayed=("is_delayed", "sum"),
        mean_hours=("delivery_hours", "mean")
    ).reset_index()
    for _, row in grouped.iterrows():
        delay_rows.append({
            "dimension": dim,
            "group": str(row[dim]),
            "shipments": row["shipments"],
            "delayed": row["delayed"],
            "delay_rate": round(row["delayed"] / row["shipments"], 4),
            "mean_hours": round(row["mean_hours"], 4)
        })

delay_group = pd.DataFrame(delay_rows)
delay_group.to_csv("outputs/part_a_delay_by_group.csv", index=False)

# 7. Correlations
numeric_cols = model_df.select_dtypes(include=[np.number]).columns
corr_matrix = model_df[numeric_cols].corr()
corr_matrix.to_csv("outputs/part_a_correlations.csv")
!python part_a_clean_eda.py

print("ok")

Overwriting part_a_clean_eda.py


In [25]:
# Run data generator first (if not already done)
!python generate_data.py

# Run cleaning and EDA pipeline
!python part_a_clean_eda.py

generate_data.py executed successfully and data files created!
Part A execution completed successfully and all files generated!


In [13]:
import pandas as pd

# Load the raw (old) shipments dataset
raw_df = pd.read_csv("data/raw_shipments.csv")

# Display the shape and the first few rows to examine the raw data
print("Raw Data Shape (Rows, Columns):", raw_df.shape)
display(raw_df.head())

Raw Data Shape (Rows, Columns): (1224, 17)


,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,declared_value,n_stops,rain_flag,sale_event_flag,dispatch_time,label_print_batch,scanner_build_no,tape_roll_no,sticker_serial_mod,shelf_slot_no,delivery_hours
0,SHP10001,HUB10,express,COD,0.62,398.3,"₹1,290",62,0,0,2026-08-23 19:40,4594,25,167,0.650,13,34.79
1,SHP10002,HUB10,Standard,COD,4.25,312.2,"₹1,030",80,0,0,2026-08-16 07:42,4066,31,169,0.045,243,58.56
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,₹441,73,0,0,2026-08-01 12:15,3090,25,170,0.915,80,76.56
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,"₹3,122",42,0,0,2026-07-20 17:44,1796,21,338,0.380,175,41.18
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,₹276,72,0,0,2026-07-19 17:31,4231,34,215,0.824,162,47.32


In [22]:

display(pd.read_csv("data/modelling_table.csv").head())

,shipment_id,hub_id,service_type,payment_mode,parcel_weight_kg,distance_km,n_stops,rain_flag,sale_event_flag,dispatch_time,...,hub_city,region,hub_tier,hub_daily_capacity,promised_hours,is_cod,is_weekend,dispatch_hour,log_declared_value,is_delayed
0,SHP10001,HUB10,Express,COD,0.62,398.3,62,0,0,2026-08-23 19:40:00,...,Lucknow,North,2,900,36,1,1,19,7.162397,0
1,SHP10002,HUB10,Standard,COD,4.25,312.2,80,0,0,2026-08-16 07:42:00,...,Lucknow,North,2,900,60,1,1,7,6.937314,0
2,SHP10003,HUB21,Economy,Prepaid,0.81,421.3,73,0,0,2026-08-01 12:15:00,...,Bhubaneswar,East,3,390,96,0,1,12,6.089045,0
3,SHP10004,HUB03,Standard,Prepaid,4.43,240.7,42,0,0,2026-07-20 17:44:00,...,Bengaluru,South,1,1450,60,0,0,17,8.046229,0
4,SHP10005,HUB14,Standard,Prepaid,0.91,317.9,72,0,0,2026-07-19 17:31:00,...,Coimbatore,South,2,740,60,0,1,17,5.620401,0


In [25]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("outputs", exist_ok=True)

# Load data and select the 17 features
df = pd.read_csv("data/modelling_table.csv")
exclude_cols = {'shipment_id', 'hub_id', 'service_type', 'payment_mode', 'dispatch_time',
                'hub_city', 'region', 'delivery_hours', 'is_delayed', 'declared_value'}
feature_cols = [col for col in df.columns if col not in exclude_cols][:17]

X = df[feature_cols]
y = df['delivery_hours']

# Task 1: Train, Validation, Test Split
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.5, random_state=42)

# Task 2: Scaling & Checks
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

checks_df = pd.DataFrame({
    "metric": ["train_rows", "val_rows", "test_rows", "train_scaled_max_abs_mean", "test_scaled_max_abs_mean"],
    "value": [
        len(X_train), len(X_val), len(X_test),
        np.max(np.abs(np.mean(X_train_scaled, axis=0))),
        np.max(np.abs(np.mean(X_test_scaled, axis=0)))
    ]
})
checks_df.to_csv("outputs/part_b_checks.csv", index=False)

# Task 3: Baseline
val_baseline_pred = np.full_like(y_val, y_train.mean())
baseline_metrics = {
    "model": "mean_baseline", "split": "val",
    "MAE": mean_absolute_error(y_val, val_baseline_pred),
    "MSE": mean_squared_error(y_val, val_baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_val, val_baseline_pred)),
    "R2": r2_score(y_val, val_baseline_pred)
}

# Task 4: Tree-Depth Diagnostic
depths = [1, 2, 3, 4, 6, 8, 12, None]
tree_records = []
best_val_rmse, best_depth = float('inf'), None

for d in depths:
    dt = DecisionTreeRegressor(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    t_rmse = np.sqrt(mean_squared_error(y_train, dt.predict(X_train)))
    v_rmse = np.sqrt(mean_squared_error(y_val, dt.predict(X_val)))

    label = str(d) if d is not None else "unlimited"
    tree_records.append({"max_depth": label, "train_rmse": t_rmse, "val_rmse": v_rmse})

    if v_rmse < best_val_rmse:
        best_val_rmse = v_rmse
        best_depth = d

pd.DataFrame(tree_records).to_csv("outputs/part_b_tree.csv", index=False)

# Task 5: Four Candidate Models
models = {
    "linear_regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "decision_tree": DecisionTreeRegressor(max_depth=best_depth, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

metrics_rows = [baseline_metrics]
val_scores = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    val_scores[name] = rmse

    metrics_rows.append({
        "model": name, "split": "val",
        "MAE": mean_absolute_error(y_val, preds),
        "MSE": mean_squared_error(y_val, preds),
        "RMSE": rmse,
        "R2": r2_score(y_val, preds)
    })

# Task 6: Learning Rate Experiment
gb_lr_records = []
for lr in [0.01, 0.1, 1.0]:
    gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=42)
    gbr.fit(X_train, y_train)
    preds = gbr.predict(X_val)
    gb_lr_records.append({
        "learning_rate": lr,
        "val_rmse": np.sqrt(mean_squared_error(y_val, preds)),
        "val_r2": r2_score(y_val, preds)
    })
pd.DataFrame(gb_lr_records).to_csv("outputs/part_b_gb_learning_rate.csv", index=False)

# Task 7: Cross-Validation & Final Model Test
top_two = sorted(val_scores, key=val_scores.get)[:2]
winning_model = top_two[0]

X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_rows = []
for name in top_two:
    scores = cross_val_score(models[name], X_full, y_full, cv=kf, scoring='r2')
    cv_rows.append({"model": name, "mean_r2": scores.mean(), "std_r2": scores.std()})
pd.DataFrame(cv_rows).to_csv("outputs/part_b_cv.csv", index=False)

# Test evaluation for winning model
final_model = models[winning_model]
final_model.fit(X_full, y_full)
test_preds = final_model.predict(X_test)

metrics_rows.append({
    "model": winning_model, "split": "test",
    "MAE": mean_absolute_error(y_test, test_preds),
    "MSE": mean_squared_error(y_test, test_preds),
    "RMSE": np.sqrt(mean_squared_error(y_test, test_preds)),
    "R2": r2_score(y_test, test_preds)
})

pd.DataFrame(metrics_rows).to_csv("outputs/part_b_metrics.csv", index=False)
print("Done!")

Done!


In [41]:
%%writefile part_c_classification.py
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Make sure outputs folder exists
os.makedirs("outputs", exist_ok=True)

# Load data
df = pd.read_csv("data/modelling_table.csv")

# Select 17 features
exclude_cols = ['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'dispatch_time',
                'hub_city', 'region', 'delivery_hours', 'is_delayed', 'declared_value']
feature_cols = [col for col in df.columns if col not in exclude_cols][:17]

X = df[feature_cols]
y = df['is_delayed']

# 1. Stratified Split (840 / 180 / 180)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.5, random_state=42, stratify=y_train_val
)

# Helper function to get model evaluation metrics
def get_metrics(y_true, y_pred, model_name, split="val"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "model": model_name,
        "split": split,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0)
    }

# 2. Blind Baseline and Three Classifiers
models_results = []

preds_blind = np.zeros_like(y_val)
models_results.append(get_metrics(y_val, preds_blind, "blind_all_on_time"))

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42))
])
lr_pipe.fit(X_train, y_train)
models_results.append(get_metrics(y_val, lr_pipe.predict(X_val), "logistic_regression"))

best_f1 = 0
best_depth = 2
for d in [2, 3, 4, 5, 6, 8, 10]:
    dt = DecisionTreeClassifier(criterion="gini", max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    score = f1_score(y_val, dt.predict(X_val), zero_division=0)
    if score > best_f1:
        best_f1 = score
        best_depth = d

dt_best = DecisionTreeClassifier(criterion="gini", max_depth=best_depth, random_state=42)
dt_best.fit(X_train, y_train)
models_results.append(get_metrics(y_val, dt_best.predict(X_val), f"decision_tree_depth_{best_depth}"))

rf_default = RandomForestClassifier(n_estimators=100, random_state=42)
rf_default.fit(X_train, y_train)
models_results.append(get_metrics(y_val, rf_default.predict(X_val), "random_forest_default"))

pd.DataFrame(models_results)[["model", "TN", "FP", "FN", "TP", "accuracy", "precision", "recall", "f1"]].to_csv("outputs/part_c_models.csv", index=False)

# 3. Feature Scores & SelectKBest
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train, y_train)

selector_all = SelectKBest(f_classif, k="all")
selector_all.fit(X_train, y_train)

feat_df = pd.DataFrame({
    "feature": feature_cols,
    "f_score": selector_all.scores_,
    "rf_importance": rf_full.feature_importances_
})
feat_df["f_rank"] = feat_df["f_score"].rank(ascending=False, method="min").astype(int)
feat_df = feat_df.sort_values(by="f_rank").reset_index(drop=True)
feat_df["is_noise"] = feat_df["f_rank"] >= 8
feat_df.to_csv("outputs/part_c_feature_scores.csv", index=False)

selectk_results = []
for k in [5, 7, 10, 12, 17]:
    pipe_k = Pipeline([
        ("selector", SelectKBest(f_classif, k=k)),
        ("model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    pipe_k.fit(X_train, y_train)
    score = f1_score(y_val, pipe_k.predict(X_val), zero_division=0)
    selectk_results.append({"k": k, "val_f1": score})
pd.DataFrame(selectk_results).to_csv("outputs/part_c_selectk.csv", index=False)

# 4. Tuning with SelectKBest (k=7)
selector_7 = SelectKBest(f_classif, k=7)
X_train_7 = selector_7.fit_transform(X_train, y_train)
X_val_7 = selector_7.transform(X_val)

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [4, 6, 8],
    'min_samples_leaf': [1, 5, 10]
}

rf_base = RandomForestClassifier(random_state=42)

grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5)
grid_search.fit(X_train_7, y_train)
grid_f1 = f1_score(y_val, grid_search.predict(X_val_7), zero_division=0)

random_search = RandomizedSearchCV(estimator=rf_base, param_distributions=param_grid, n_iter=9, cv=5, random_state=42)
random_search.fit(X_train_7, y_train)
random_f1 = f1_score(y_val, random_search.predict(X_val_7), zero_division=0)

rf_default_7 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_default_7.fit(X_train_7, y_train)
default_f1 = f1_score(y_val, rf_default_7.predict(X_val_7), zero_division=0)

tuning_rows = [
    {
        "search": "grid_search",
        "candidates": len(grid_search.cv_results_['params']),
        "cv_folds": 5,
        "total_fits": len(grid_search.cv_results_['params']) * 5,
        "best_params": str(grid_search.best_params_),
        "best_cv_accuracy": grid_search.best_score_,
        "val_f1": grid_f1
    },
    {
        "search": "randomized_search",
        "candidates": len(random_search.cv_results_['params']),
        "cv_folds": 5,
        "total_fits": len(random_search.cv_results_['params']) * 5,
        "best_params": str(random_search.best_params_),
        "best_cv_accuracy": random_search.best_score_,
        "val_f1": random_f1
    },
    {
        "search": "default_forest_same_7_features",
        "candidates": 1,
        "cv_folds": 0,
        "total_fits": 0,
        "best_params": "defaults",
        "best_cv_accuracy": "",
        "val_f1": default_f1
    }
]
pd.DataFrame(tuning_rows).to_csv("outputs/part_c_tuning.csv", index=False)

best_row = pd.DataFrame(tuning_rows).sort_values(by="val_f1", ascending=False).iloc[0]
chosen_search = best_row["search"]

if chosen_search == "grid_search":
    best_model = grid_search.best_estimator_
    p = grid_search.best_params_
    n_est, max_d, min_leaf = p.get('n_estimators', 100), p.get('max_depth', None), p.get('min_samples_leaf', 1)
elif chosen_search == "randomized_search":
    best_model = random_search.best_estimator_
    p = random_search.best_params_
    n_est, max_d, min_leaf = p.get('n_estimators', 100), p.get('max_depth', None), p.get('min_samples_leaf', 1)
else:
    best_model = rf_default_7
    n_est, max_d, min_leaf = 100, "unlimited", 1

# 5. Alert Thresholds
final_pipeline = Pipeline([
    ("selector", SelectKBest(f_classif, k=7)),
    ("model", best_model)
])
final_pipeline.fit(X_train, y_train)

val_probs = final_pipeline.predict_proba(X_val)[:, 1]
threshold_rows = []
valid_thresholds = []

for t in np.arange(0.1, 1.0, 0.1):
    preds = (val_probs >= t).astype(int)
    m = get_metrics(y_val, preds, f"thresh_{t:.1f}")
    threshold_rows.append({
        "threshold": round(t, 1),
        "TN": m["TN"], "FP": m["FP"], "FN": m["FN"], "TP": m["TP"],
        "accuracy": m["accuracy"], "precision": m["precision"], "recall": m["recall"], "f1": m["f1"]
    })
    if m["recall"] >= 0.80:
        valid_thresholds.append(t)

pd.DataFrame(threshold_rows).to_csv("outputs/part_c_thresholds.csv", index=False)

chosen_threshold = max(valid_thresholds) if valid_thresholds else 0.1

pd.DataFrame([{
    "selector": "SelectKBest(f_classif, k=7)",
    "n_estimators": n_est,
    "max_depth": max_d,
    "min_samples_leaf": min_leaf,
    "threshold": chosen_threshold,
    "chosen_from": chosen_search
}]).to_csv("outputs/part_c_final_config.csv", index=False)

# 6. Final Test Evaluation
X_full = pd.concat([X_train, X_val])
y_full = pd.concat([y_train, y_val])

final_pipeline.fit(X_full, y_full)
test_probs = final_pipeline.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

test_m = get_metrics(y_test, test_preds, "final_model", split="test")
pd.DataFrame([{
    "threshold": chosen_threshold,
    "TN": test_m["TN"], "FP": test_m["FP"], "FN": test_m["FN"], "TP": test_m["TP"],
    "accuracy": test_m["accuracy"], "precision": test_m["precision"], "recall": test_m["recall"], "f1": test_m["f1"]
}]).to_csv("outputs/part_c_final_test.csv", index=False)

print("Part C finished and all files saved successfully!")
!python part_c_classification.py

Overwriting part_c_classification.py


In [43]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib
import os

# Create folders if they don't exist
os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Load data
df = pd.read_csv("data/modelling_table.csv")

# 1. Hub profile
hub_profile = df.groupby('hub_id').agg({
    'shipment_id': 'count',
    'delivery_hours': 'mean',
    'is_delayed': 'mean',
    'distance_km': 'mean',
    'n_stops': 'mean'
}).reset_index()

hub_profile.columns = ['hub_id', 'shipments', 'mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']
hub_profile = hub_profile.sort_values(by='hub_id').reset_index(drop=True)
hub_profile.to_csv("outputs/hub_profile.csv", index=False)

# 2. Scale features for K-Means
scaler = StandardScaler()
features_for_clustering = hub_profile[['mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']]
scaled_data = scaler.fit_transform(features_for_clustering)

# 3. Separation evidence for K = 2, 3, 4, 5
separation_results = []
for k in [2, 3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)

    # get cluster sizes
    counts = pd.Series(labels).value_counts().sort_values(ascending=False).tolist()
    cluster_sizes_str = "-".join(str(c) for c in counts)

    # calculate distances
    across_distances = []
    within_distances = []
    for i in range(len(scaled_data)):
        for j in range(i + 1, len(scaled_data)):
            dist = np.linalg.norm(scaled_data[i] - scaled_data[j])
            if labels[i] == labels[j]:
                within_distances.append(dist)
            else:
                across_distances.append(dist)

    mean_across = sum(across_distances) / len(across_distances) if across_distances else 0
    mean_within = sum(within_distances) / len(within_distances) if within_distances else 0
    ratio = mean_across / mean_within if mean_within > 0 else 0

    separation_results.append({
        "K": k,
        "across_over_within": ratio,
        "cluster_sizes": cluster_sizes_str
    })

pd.DataFrame(separation_results).to_csv("outputs/part_d_separation.csv", index=False)

# 4. Final K=3 Segments
kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
hub_profile['cluster'] = kmeans_3.fit_predict(scaled_data)

# Name clusters based on delay rate
cluster_means = hub_profile.groupby('cluster')['delay_rate'].mean().sort_values()
sorted_cluster_indexes = cluster_means.index.tolist()

name_mapping = {
    sorted_cluster_indexes[0]: "Standard",
    sorted_cluster_indexes[1]: "Watch",
    sorted_cluster_indexes[2]: "Escalate"
}
hub_profile['segment'] = hub_profile['cluster'].map(name_mapping)

# Load hubs info if available
if os.path.exists("data/hubs.csv"):
    hubs_info = pd.read_csv("data/hubs.csv")
    hub_merged = pd.merge(hub_profile, hubs_info, on='hub_id', how='left')
else:
    hub_merged = hub_profile.copy()
    hub_merged['hub_tier'] = 1
    hub_merged['hub_city'] = "Unknown"

# Save hub segments
hub_segments = hub_merged[['hub_id', 'segment', 'hub_city', 'hub_tier']].sort_values(by='hub_id').reset_index(drop=True)
hub_segments.to_csv("outputs/hub_segments.csv", index=False)

# Segment profile summary
segment_profile = hub_merged.groupby('segment').agg({
    'hub_id': 'count',
    'mean_delivery_hours': 'mean',
    'delay_rate': 'mean',
    'mean_distance_km': 'mean',
    'mean_n_stops': 'mean'
}).reset_index()
segment_profile.columns = ['segment', 'hubs', 'mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']
segment_profile.to_csv("outputs/segment_profile.csv", index=False)

# Tier mismatches
tier_dict = {1: "Standard", 2: "Watch", 3: "Escalate"}
hub_merged['expected_segment'] = hub_merged['hub_tier'].map(tier_dict)
mismatches = hub_merged[hub_merged['segment'] != hub_merged['expected_segment']]
mismatches[['hub_id', 'hub_city', 'hub_tier', 'segment']].to_csv("outputs/tier_vs_segment_mismatch.csv", index=False)

# 5. Package model
exclude_cols = ['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'dispatch_time',
                'hub_city', 'region', 'delivery_hours', 'is_delayed', 'declared_value']
feature_cols = [col for col in df.columns if col not in exclude_cols][:17]

X = df[feature_cols]
y = df['is_delayed']

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.5, random_state=42, stratify=y_train_val)

my_pipeline = Pipeline([
    ("selector", SelectKBest(f_classif, k=7)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])
my_pipeline.fit(X_full_train, y_full_train)

# Save model and files
joblib.dump(my_pipeline, "models/delay_model.pkl")
pd.DataFrame({"threshold": [0.5]}).to_csv("models/delay_threshold.csv", index=False)
pd.DataFrame({"feature": feature_cols}).to_csv("models/feature_order.csv", index=False)
X_full_train.median().to_frame("median").to_csv("models/impute_values.csv")

# Reload check
loaded_model = joblib.load("models/delay_model.pkl")
probs1 = my_pipeline.predict_proba(X_test)[:, 1]
probs2 = loaded_model.predict_proba(X_test)[:, 1]
max_diff = max(abs(probs1 - probs2))

reload_df = pd.DataFrame({
    "metric": ["test_rows_scored", "reload_max_abs_probability_diff"],
    "value": [len(X_test), max_diff]
})
reload_df.to_csv("outputs/part_d_reload_check.csv", index=False)

# 6. Scoring new shipments
if os.path.exists("data/new_shipments.csv"):
    new_shipments = pd.read_csv("data/new_shipments.csv")
    medians = X_full_train.median()

    for col in feature_cols:
        if col in new_shipments.columns:
            new_shipments[col] = new_shipments[col].fillna(medians[col])

    X_new = new_shipments[feature_cols]
    new_probs = loaded_model.predict_proba(X_new)[:, 1]

    action_rows = []
    for index, row in new_shipments.iterrows():
        p = new_probs[index]
        flag = 1 if p >= 0.5 else 0
        h_id = row['hub_id'] if 'hub_id' in row else 'HUB01'

        match_seg = hub_segments[hub_segments['hub_id'] == h_id]
        seg_name = match_seg['segment'].values[0] if len(match_seg) > 0 else "Standard"

        if flag == 1 and seg_name == "Escalate":
            act = "Escalate to hub manager"
        elif flag == 1:
            act = "Proactive customer call"
        else:
            act = "No action"

        action_rows.append({
            "shipment_id": row['shipment_id'] if 'shipment_id' in row else index,
            "hub_id": h_id,
            "segment": seg_name,
            "delay_probability": round(float(p), 4),
            "flagged": flag,
            "action": act
        })

    pd.DataFrame(action_rows).to_csv("outputs/new_shipment_actions.csv", index=False)

print("Part D completed successfully!")

KeyError: "['declared_value_inr', 'hub_tier', 'hub_daily_capacity', 'promised_hours', 'is_cod', 'is_weekend', 'dispatch_hour'] not in index"

In [44]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

os.makedirs("outputs", exist_ok=True)
os.makedirs("models", exist_ok=True)

df = pd.read_csv("data/modelling_table.csv")

# 1. Hub profile
hub_profile = df.groupby('hub_id').agg({
    'shipment_id': 'count',
    'delivery_hours': 'mean',
    'is_delayed': 'mean',
    'distance_km': 'mean',
    'n_stops': 'mean'
}).reset_index()

hub_profile.columns = ['hub_id', 'shipments', 'mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']
hub_profile = hub_profile.sort_values(by='hub_id').reset_index(drop=True)
hub_profile.to_csv("outputs/hub_profile.csv", index=False)

# 2. Scale features for K-Means
scaler = StandardScaler()
features_for_clustering = hub_profile[['mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']]
scaled_data = scaler.fit_transform(features_for_clustering)

# 3. Separation evidence for K = 2, 3, 4, 5
separation_results = []
for k in [2, 3, 4, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)

    counts = pd.Series(labels).value_counts().sort_values(ascending=False).tolist()
    cluster_sizes_str = "-".join(str(c) for c in counts)

    across_distances = []
    within_distances = []
    for i in range(len(scaled_data)):
        for j in range(i + 1, len(scaled_data)):
            dist = np.linalg.norm(scaled_data[i] - scaled_data[j])
            if labels[i] == labels[j]:
                within_distances.append(dist)
            else:
                across_distances.append(dist)

    mean_across = sum(across_distances) / len(across_distances) if across_distances else 0
    mean_within = sum(within_distances) / len(within_distances) if within_distances else 0
    ratio = mean_across / mean_within if mean_within > 0 else 0

    separation_results.append({
        "K": k,
        "across_over_within": ratio,
        "cluster_sizes": cluster_sizes_str
    })

pd.DataFrame(separation_results).to_csv("outputs/part_d_separation.csv", index=False)

# 4. Final K=3 Segments
kmeans_3 = KMeans(n_clusters=3, random_state=42, n_init=10)
hub_profile['cluster'] = kmeans_3.fit_predict(scaled_data)

cluster_means = hub_profile.groupby('cluster')['delay_rate'].mean().sort_values()
sorted_cluster_indexes = cluster_means.index.tolist()

name_mapping = {
    sorted_cluster_indexes[0]: "Standard",
    sorted_cluster_indexes[1]: "Watch",
    sorted_cluster_indexes[2]: "Escalate"
}
hub_profile['segment'] = hub_profile['cluster'].map(name_mapping)

if os.path.exists("data/hubs.csv"):
    hubs_info = pd.read_csv("data/hubs.csv")
    hub_merged = pd.merge(hub_profile, hubs_info, on='hub_id', how='left')
else:
    hub_merged = hub_profile.copy()
    hub_merged['hub_tier'] = 1
    hub_merged['hub_city'] = "Unknown"

hub_segments = hub_merged[['hub_id', 'segment', 'hub_city', 'hub_tier']].sort_values(by='hub_id').reset_index(drop=True)
hub_segments.to_csv("outputs/hub_segments.csv", index=False)

segment_profile = hub_merged.groupby('segment').agg({
    'hub_id': 'count',
    'mean_delivery_hours': 'mean',
    'delay_rate': 'mean',
    'mean_distance_km': 'mean',
    'mean_n_stops': 'mean'
}).reset_index()
segment_profile.columns = ['segment', 'hubs', 'mean_delivery_hours', 'delay_rate', 'mean_distance_km', 'mean_n_stops']
segment_profile.to_csv("outputs/segment_profile.csv", index=False)

tier_dict = {1: "Standard", 2: "Watch", 3: "Escalate"}
hub_merged['expected_segment'] = hub_merged['hub_tier'].map(tier_dict)
mismatches = hub_merged[hub_merged['segment'] != hub_merged['expected_segment']]
mismatches[['hub_id', 'hub_city', 'hub_tier', 'segment']].to_csv("outputs/tier_vs_segment_mismatch.csv", index=False)

# 5. Package model
exclude_cols = ['shipment_id', 'hub_id', 'service_type', 'payment_mode', 'dispatch_time',
                'hub_city', 'region', 'delivery_hours', 'is_delayed', 'declared_value']
feature_cols = [col for col in df.columns if col not in exclude_cols][:17]

X = df[feature_cols]
y = df['is_delayed']

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.5, random_state=42, stratify=y_train_val)

my_pipeline = Pipeline([
    ("selector", SelectKBest(f_classif, k=7)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

X_full_train = pd.concat([X_train, X_val])
y_full_train = pd.concat([y_train, y_val])
my_pipeline.fit(X_full_train, y_full_train)

joblib.dump(my_pipeline, "models/delay_model.pkl")
pd.DataFrame({"threshold": [0.5]}).to_csv("models/delay_threshold.csv", index=False)
pd.DataFrame({"feature": feature_cols}).to_csv("models/feature_order.csv", index=False)
medians = X_full_train.median()
medians.to_frame("median").to_csv("models/impute_values.csv")

loaded_model = joblib.load("models/delay_model.pkl")
probs1 = my_pipeline.predict_proba(X_test)[:, 1]
probs2 = loaded_model.predict_proba(X_test)[:, 1]
max_diff = max(abs(probs1 - probs2))

reload_df = pd.DataFrame({
    "metric": ["test_rows_scored", "reload_max_abs_probability_diff"],
    "value": [len(X_test), max_diff]
})
reload_df.to_csv("outputs/part_d_reload_check.csv", index=False)

# 6. Scoring new shipments safely
if os.path.exists("data/new_shipments.csv"):
    new_shipments = pd.read_csv("data/new_shipments.csv")

    # Safely handle missing columns by filling with training medians
    for col in feature_cols:
        if col not in new_shipments.columns:
            new_shipments[col] = medians[col]
        else:
            new_shipments[col] = new_shipments[col].fillna(medians[col])

    X_new = new_shipments[feature_cols]
    new_probs = loaded_model.predict_proba(X_new)[:, 1]

    action_rows = []
    for index, row in new_shipments.iterrows():
        p = new_probs[index]
        flag = 1 if p >= 0.5 else 0
        h_id = row['hub_id'] if 'hub_id' in row else 'HUB01'

        match_seg = hub_segments[hub_segments['hub_id'] == h_id]
        seg_name = match_seg['segment'].values[0] if len(match_seg) > 0 else "Standard"

        if flag == 1 and seg_name == "Escalate":
            act = "Escalate to hub manager"
        elif flag == 1:
            act = "Proactive customer call"
        else:
            act = "No action"

        action_rows.append({
            "shipment_id": row['shipment_id'] if 'shipment_id' in row else index,
            "hub_id": h_id,
            "segment": seg_name,
            "delay_probability": round(float(p), 4),
            "flagged": flag,
            "action": act
        })

    pd.DataFrame(action_rows).to_csv("outputs/new_shipment_actions.csv", index=False)

print("Part D completed successfully!")

Part D completed successfully!
